# Módulo 5: Regressão Linear Múltipla e Padronização de Dados

> **Resumo Didático Integrado** — Baseado no *Capítulo 3 (Análise de Dados — Seção 2: Regressão Múltipla & Padronização)*.
> 
> Este caderno é **100% autocontido**: você aprenderá a modelar fenômenos complexos envolvendo múltiplos preditores simultâneos, diagnosticar multicolinearidade (**VIF**), incorporar relações não lineares via termos polinomiais (`I(X^2)`) e dominar o papel fundamental da **Padronização de Dados (Z-Score vs Min-Max)** para comparar o impacto real de cada variável independente.

---

## 1. Visão Geral e Fundamentação Matemática

### 1.1 A Equação da Regressão Linear Múltipla
Na maioria dos cenários reais, um único fator é insuficiente para explicar a variabilidade de um fenômeno. A regressão linear múltipla expande a equação linear para $k$ variáveis preditoras:

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_k X_k + \epsilon$$

Onde cada coeficiente parcial $\beta_j$ representa o impacto de $X_j$ sobre $Y$, **mantendo todas as demais variáveis constantes** (*ceteris paribus*).

### 1.2 O Coeficiente de Determinação Ajustado ($R^2_{adj}$)
Diferente da regressão simples, na regressão múltipla o $R^2$ comum **nunca diminui** quando adicionamos novas variáveis — mesmo que sejam variáveis aleatórias irrelevantes. Por isso, utilizamos o **$R^2$ Ajustado**:

$$R^2_{adj} = 1 - \left[ \frac{(1 - R^2)(n - 1)}{n - k - 1} \right]$$

Onde $n$ é o número de observações e $k$ é a quantidade de preditores. O $R^2_{adj}$ penaliza a inclusão de variáveis desnecessárias e só aumenta se o novo preditor realmente melhorar o poder explicativo do modelo além do mero acaso.

### 1.3 Diagnóstico de Multicolinearidade (VIF)
A **Multicolinearidade** ocorre quando duas ou mais variáveis preditoras são fortemente correlacionadas entre si. Isso infla a variância das estimativas dos coeficientes, tornando-os instáveis. O indicador padrão de diagnóstico é o **Fator de Inflação da Variância (VIF - *Variance Inflation Factor*)**:

$$VIF_j = \frac{1}{1 - R_j^2}$$

* $VIF \approx 1$: Ausência de colinearidade.
* $VIF > 5$: Alerta moderado de multicolinearidade.
* $VIF > 10$: Multicolinearidade severa (recomenda-se remover ou combinar os preditores colineares).

---

## 2. A Importância Fundamental da Padronização de Dados

Em dados brutos, variáveis possuem ordens de grandeza completamente distintas (ex.: `população` na casa dos milhões vs. `analfabetismo` na casa de decimais entre 0,5% e 2,8%). 
* **Problema**: O valor numérico do coeficiente bruto $\beta$ depende da unidade de medida. Um coeficiente pequeno pode pertencer a uma variável crucial apenas porque sua unidade é gigante.
* **Solução**: Padronizar as variáveis para uma escala adimensional comum.

### 2.1 Técnicas de Padronização
1. **Z-Score (Z-Norm / Padronização Normal)**:
   $$z = \frac{x - \bar{x}}{\sigma}$$
   Centraliza os dados na média $0$ com desvio padrão $1$. Preserva a forma da distribuição e os outliers.
2. **Min-Max Scaling (Normalização por Amplitude)**:
   $$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$
   Comprime todos os valores estritamente dentro do intervalo $[0, 1]$.

### 2.2 Coeficientes Padronizados (Beta Padronizado $\beta^*$)
Ao ajustar a regressão com variáveis padronizadas por Z-Score, **todos os coeficientes passam a ser expressos em unidades de desvio padrão**. Isso permite ranquear diretamente quais variáveis exercem a maior influência sobre a resposta $Y$.

---

## 3. Passo a Passo Prático: Modelagem Múltipla e Padronização

Aplicaremos a metodologia didática:
* 🎯 **O que fazer**
* 💻 **Como fazer**
* 💡 **Por que fazer**

### Passo 1: Ingestão de Dados e Matriz de Correlação Multivariada

🎯 **O que fazer**:
Limpar a sessão, carregar a base socioeconômica dos estados americanos (`datasets/assassinato_EUA_1977.RData`), inspecionar a matriz de correlação entre os preditores e a taxa de assassinatos (`assassinato`), e gerar visualização gráfica com mapa de calor.

💻 **Como fazer**:

In [ ]:
# Limpeza do ambiente
rm(list = ls())

# Configuração de gráficos no Jupyter
if (requireNamespace("repr", quietly = TRUE)) {
  options(repr.plot.width = 11, repr.plot.height = 6)
}

caminho_dados <- if (file.exists("../datasets/assassinato_EUA_1977.RData")) {
  "../datasets/assassinato_EUA_1977.RData"
} else if (file.exists("datasets/assassinato_EUA_1977.RData")) {
  "datasets/assassinato_EUA_1977.RData"
} else {
  stop("Arquivo de dados não encontrado!")
}

load(caminho_dados) # Carrega o data frame 'st'
cat("Base 'st' carregada. Estados:", nrow(st), "| Variáveis:", ncol(st), "\n")

# Seleção das colunas numéricas para análise de correlação
colunas_num <- c("assassinato", "populacao", "analfabetismo", "renda", "expectativa", "congelamento", "ensino_medio")
mat_cor <- cor(st[, colunas_num])

cat("\nCorrelações lineares com a taxa de homicídios (assassinato):\n")
print(round(mat_cor["assassinato", ], 3))

# Visualização gráfica da matriz de correlação
if (requireNamespace("corrplot", quietly = TRUE)) {
  library(corrplot)
  corrplot(mat_cor, method = "color", type = "upper",
           addCoef.col = "black", tl.col = "black", tl.srt = 45,
           title = "Matriz de Correlação Multivariada", mar = c(0,0,2,0))
} else {
  # Fallback gráfico base R
  heatmap(mat_cor, main = "Mapa de Correlação", col = heat.colors(12))
}

💡 **Por que fazer**:
A matriz de correlação identifica precocemente quais atributos possuem forte ligação com a variável alvo (ex.: `analfabetismo` positivamente correlacionado com `assassinato`, e `expectativa` fortemente inversa), além de alertar sobre possíveis colinearidades entre preditores antes do ajuste do modelo.

### Passo 2: Construção do Modelo Padrão-Base (*Baseline*) e Seleção de Atributos

🎯 **O que fazer**:
Ajustar um modelo *Baseline* com todos os atributos disponíveis e, em seguida, construir um modelo parcimonioso removendo atributos redundantes ou sem significância estatística ($p > 0.05$), comparando o $R^2$ e o $R^2$ ajustado.

💻 **Como fazer**:

In [ ]:
# 1. Modelo Baseline (todas as variáveis numéricas)
mod_base <- lm(assassinato ~ populacao + analfabetismo + renda + expectativa + congelamento + ensino_medio, data = st)
res_base <- summary(mod_base)

# 2. Modelo Refinado (selecionando preditores estatisticamente relevantes)
# Removemos 'renda' e 'ensino_medio' que apresentam p-valores elevados e redundância
mod_refinado <- lm(assassinato ~ populacao + analfabetismo + congelamento + expectativa, data = st)
res_refinado <- summary(mod_refinado)

cat("--- Comparação de Desempenho ---\n")
cat("Modelo Baseline:\n")
cat("  R² Comum:", round(res_base$r.squared, 4), "| R² Ajustado:", round(res_base$adj.r.squared, 4), "\n")
cat("Modelo Refinado (Mais Parcimonioso):\n")
cat("  R² Comum:", round(res_refinado$r.squared, 4), "| R² Ajustado:", round(res_refinado$adj.r.squared, 4), "\n\n")

print(summary(mod_refinado))

💡 **Por que fazer**:
O princípio da parcimônia (Navalha de Ockham) preconiza que modelos mais simples com menor número de parâmetros generalizam melhor para novos dados. Observe como o $R^2$ ajustado do modelo refinado se mantém equivalente ou superior ao baseline, eliminando o risco de sobreajuste (*overfitting*).

### Passo 3: Avaliação de Multicolinearidade via VIF

🎯 **O que fazer**:
Calcular o Fator de Inflação da Variância (VIF) para cada preditor do modelo refinado para garantir que não há redundância matemática degradando as estimativas.

💻 **Como fazer**:

In [ ]:
# Diagnóstico de VIF utilizando o pacote 'car' ou cálculo matricial direto
if (requireNamespace("car", quietly = TRUE)) {
  vif_valores <- car::vif(mod_refinado)
  cat("Fatores de Inflação da Variância (VIF) via pacote car:\n")
  print(round(vif_valores, 2))
} else {
  # Cálculo analítico do VIF manual: VIF_j = 1 / (1 - R_j^2)
  vif_manual <- numeric(4)
  names(vif_manual) <- c("populacao", "analfabetismo", "congelamento", "expectativa")
  vif_manual["populacao"]    <- 1 / (1 - summary(lm(populacao ~ analfabetismo + congelamento + expectativa, data = st))$r.squared)
  vif_manual["analfabetismo"]<- 1 / (1 - summary(lm(analfabetismo ~ populacao + congelamento + expectativa, data = st))$r.squared)
  vif_manual["congelamento"] <- 1 / (1 - summary(lm(congelamento ~ populacao + analfabetismo + expectativa, data = st))$r.squared)
  vif_manual["expectativa"]  <- 1 / (1 - summary(lm(expectativa ~ populacao + analfabetismo + congelamento, data = st))$r.squared)
  cat("Fatores de Inflação da Variância (VIF) calculados manualmente:\n")
  print(round(vif_manual, 2))
}

cat("\nDiagnóstico: Todos os valores de VIF são substancialmente inferiores a 5.\n")
cat("Conclusão: O modelo está livre de multicolinearidade prejudicial!\n")

💡 **Por que fazer**:
Preditores colineares mascaram a significância uns dos outros (podendo transformar variáveis teoricamente vitais em variáveis aparentemente insignificantes no teste $t$). Confirmar que o VIF $< 5$ assegura que cada coeficiente reflete uma contribuição única e estável.

### Passo 4: Implementação e Análise da Padronização de Dados (Z-Score)

🎯 **O que fazer**:
Aplicar a padronização Z-Score nos dados, ajustar o mesmo modelo sobre as variáveis padronizadas e comparar os coeficientes brutos versus os coeficientes padronizados ($eta^*$) para ranquear o real impacto relativo de cada variável.

💻 **Como fazer**:

In [ ]:
# Criando cópia padronizada das variáveis numéricas
st_padronizado <- as.data.frame(scale(st[, colunas_num]))

# Ajustando o modelo linear idêntico sobre dados padronizados (sem intercepto explícito, pois a média de Z é 0)
mod_padronizado <- lm(assassinato ~ populacao + analfabetismo + congelamento + expectativa, data = st_padronizado)

# Extração e comparação direta dos coeficientes
coef_bruto       <- coef(mod_refinado)[-1]       # exclui intercepto bruto
coef_padronizado <- coef(mod_padronizado)[-1]   # coeficientes beta*

tabela_impacto <- data.frame(
  Coef_Bruto = round(coef_bruto, 5),
  Beta_Padronizado = round(coef_padronizado, 4),
  Impacto_Relativo_Absoluto = round(abs(coef_padronizado), 4)
)
# Ordenando por ordem decrescente de importância real
tabela_impacto <- tabela_impacto[order(-tabela_impacto$Impacto_Relativo_Absoluto), ]

print("--- Comparação de Coeficientes Brutos vs. Padronizados ---")
print(tabela_impacto)

💡 **Por que fazer**:
Observe o coeficiente bruto da `populacao`: ele é minúsculo ($0.00022$), enquanto o de `analfabetismo` é $1.86$. Olhando apenas os valores brutos, alguém erroneamente concluiria que a população não tem relevância! Porém, ao analisar o **Beta Padronizado**, vemos que a população tem um peso expressivo ($+0.23$), e que `expectativa de vida` ($-0.48$) e `analfabetismo` ($+0.35$) são as forças dominantes. A padronização elimina a ilusão das unidades de medida.

### Passo 5: Termos Polinomiais e Diagnóstico de Resíduos

🎯 **O que fazer**:
Avaliar a inclusão de termos de potência não lineares utilizando o operador de isolamento `I(X^2)` e validar graficamente os resíduos do modelo padronizado.

💻 **Como fazer**:

In [ ]:
# Testando modelo com termo quadrático para população: I(populacao^2)
mod_poli <- lm(assassinato ~ populacao + I(populacao^2) + analfabetismo + congelamento + expectativa, data = st_padronizado)
print("Resumo do modelo polinomial:")
print(summary(mod_poli)$coefficients)

# Diagnóstico de resíduos do modelo padronizado final
residuos_pad <- residuals(mod_padronizado)
ajustados_pad <- fitted(mod_padronizado)

par(mfrow = c(1, 2))
# Dispersão de resíduos vs ajustados
plot(ajustados_pad, residuos_pad,
     main = "Resíduos vs Valores Ajustados",
     xlab = "Taxa Predita Padronizada", ylab = "Resíduos",
     pch = 19, col = "#1d3557")
abline(h = 0, col = "red", lty = 2, lwd = 2)

# Normal Q-Q Plot para checar normalidade dos erros
qqnorm(residuos_pad, pch = 19, col = "#457b9d", main = "Normal Q-Q Plot dos Erros")
qqline(residuos_pad, col = "red", lwd = 2)
par(mfrow = c(1, 1))

💡 **Por que fazer**:
O operador `I()` é indispensável em fórmulas R porque o símbolo `^` dentro de fórmulas normalmente expressa interações entre termos. Usar `I(X^2)` força o cálculo aritmético da potência. Já os gráficos de diagnóstico confirmam se os erros seguem distribuição normal (alinhados na reta Q-Q), validando a confiabilidade das inferências.

## 4. Conclusão e Próximos Passos
Neste Módulo 5, consolidamos:
* A formulação da **Regressão Linear Múltipla** e o papel do $R^2$ ajustado.
* Diagnóstico de **Multicolinearidade via VIF**.
* O impacto definitivo da **Padronização Z-Score** na geração de coeficientes padronizados ($eta^*$) comparáveis.
* Modelagem de curvaturas com **termos polinomiais `I(X^2)`**.

Até aqui, modelamos variáveis contínuas (preços, taxas de homicídio). No **Módulo 6 (`06_Classificacao_Regressao_Logistica_e_Metricas.ipynb`)**, entraremos no domínio dos problemas de **Classificação Binária** (0 ou 1, Potável ou Não Potável) utilizando a **Regressão Logística**, **Matriz de Confusão**, **Log-Loss** e a **Curva ROC / AUC**.